# Phase 1B.3 — completion only, immutable CORE reuse
This notebook does **not** execute a CORE sweep. It imports SHA256-verified completed 1,734 local rows, resumes only remaining MP/external/boundary work and writes `reference_v3_colab/`. No Domain v1/tolerance change, neural training, or training-label generation. The legacy Phase1B.2 notebook filename now routes to this completion workflow; its original is archived.

Before running, sync the new repository source and `reference_v3/immutable/` snapshot to the selected GitHub ref. A missing/mismatched baseline **aborts**, rather than triggering a new sweep. Complete MP/PDE/boundary work may take hours; persistent Drive checkpoints are recommended.


In [2]:
from pathlib import Path
import os, subprocess, sys, json, atexit, shutil, platform
REPO_URL='https://github.com/WonC-Lab/MartingaleONet.git'
GIT_REF='main'  # Change only to your published branch/commit containing this workflow.
REPO=Path('/content/MartingaleONet')
USE_DRIVE=True  # False uses ephemeral /content output; download the ZIP before reset.
if not REPO.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
    subprocess.run(['git','checkout',GIT_REF],cwd=REPO,check=True)
os.chdir(REPO)
assert (REPO/'reference_v3/immutable/LOCAL_CORE_MANIFEST.json').is_file(), 'Publish the completed immutable CORE snapshot first; no automatic CORE rerun.'
assert (REPO/'reference_v3/reuse.py').is_file(), 'Selected GitHub ref does not contain Phase1B.3 modules.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/MartingaleONet/reference_v3_colab')
else:
    OUT=REPO/'reference_v3_colab'
OUT.mkdir(parents=True,exist_ok=True)
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
os.environ.pop('MPMATH_NOGMPY',None)
subprocess.run([sys.executable,'-m','pip','install','-r','reference_v3/requirements-colab.txt'],check=True)
with (OUT/'ENVIRONMENT.txt').open('w') as h:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=h,check=True)
(OUT/'HOST.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'source_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'execution_location':'Google Colab'},indent=2))
WORKERS=min(4,os.cpu_count() or 2)  # Increase only if RAM/CPU permits.
def package():
    return shutil.make_archive(str(OUT.parent/'reference_v3_colab_results'),'zip',root_dir=OUT.parent,base_dir=OUT.name)
atexit.register(package)
def run(module,*args):
    label=module.rsplit('.',1)[-1]
    with (OUT/(label+'_execution.log')).open('a') as log:
        result=subprocess.run([sys.executable,'-u','-m',module,*args],stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((OUT/(label+'_execution.log')).read_text()[-5000:])
        package()
        raise RuntimeError(module+' failed; completed raw checkpoints and ZIP preserved')
    print(module,'completed; raw output/log saved')


Mounted at /content/drive


## Validate and reuse completed CORE — zero sweep calls
The import checks archive bytes, expanded raw bytes, exact original input catalog and canonical pricing-module hashes. On rerun, output bytes must match; it never overwrites a different CORE file. The following cell contains **no** `coverage` command. Original local files remain read only.


In [3]:
run('pytest','reference_v3/tests','reference_v2/tests','-q','--junitxml='+str(OUT/'tests.xml'))
run('reference_v3.reuse','import','--output',str(OUT))
receipt=json.loads((OUT/'CORE_REUSE_RECEIPT.json').read_text())
assert receipt['reused']==1734 and receipt['recomputed']==0
print(receipt)


pytest completed; raw output/log saved
reference_v3.reuse completed; raw output/log saved
{'reused': 1734, 'recomputed': 0, 'source_manifest': 'reference_v3/immutable/LOCAL_CORE_MANIFEST.json', 'verified_uncompressed_sha256': '3ba4ff1aea78c6d53b026c8287ba8c006e8bfd8148423848d957a0704e868220', 'deliberate_environment_cross_validation_subset': [], 'pricing_calls_in_import': 0}


## Remaining CORE MP/FD/strike/COS escalation
All retained failures/near-failures stay in the denominator. Checkpoints and validated completed point evidence are reused; only pending inputs are calculated. This reprices individual difficult inputs as required numerical escalation, **not** a repeated A/B full sweep. Independent 50/80/120-digit, cutoff/degree, physical FD and strike evidence are saved per input.


In [4]:
run('reference_v3.escalation','--scope','all','--workers',str(WORKERS),'--output',str(OUT))
run('reference_v3.review','--output',str(OUT))


reference_v3.escalation completed; raw output/log saved
reference_v3.review completed; raw output/log saved


## Deliberate external cross-environment subset — six CORE inputs plus two exact-zero challenges
Exactly eight predeclared aligned cases: ATM, deep OTM, deep ITM, short CORE, high sigma, near-Feller, severe exact v0=0, typical exact v0=0. This is explicitly a cross-environment subset and mandatory external triangulation; the other CORE inputs are not reswept. Exact-zero QuantLib model rejection is preserved, never replaced silently by epsilon variance. Engine setting checkpoints are namespaced by environment.


In [5]:
run('reference_v3.external','--cross-environment','--refine','--output',str(OUT))
run('reference_v3.review','--output',str(OUT))


reference_v3.external completed; raw output/log saved
reference_v3.review completed; raw output/log saved


## Only the training-required boundary/collar set
Resume the original deterministic finite x=+/-1.2, v_max=.25 and v=0 inputs on CORE parameter/maturity support. Already completed local boundary rows are imported. Evaluation-only OOD is not expanded merely to produce training labels. Limiting-PDE checks are diagnostic until the original protocol's convergence obligations are met; no new residual tolerance is invented.


In [ ]:
run('reference_v3.boundary','--workers',str(WORKERS),'--output',str(OUT))
run('reference_v3.escalation','--scope','boundary','--workers',str(WORKERS),'--output',str(OUT))
run('reference_v3.continuity','--output',str(OUT))
run('reference_v3.boundary_pde','--workers',str(WORKERS),'--output',str(OUT))


reference_v3.boundary completed; raw output/log saved


## Final review, all raw figures/reports and artifact export
No gate upgrade from execution count alone. Unresolved numerical evidence remains FAIL. A certified engine lock is created only when every original CORE requirement passes; no neural labels or training are generated. All artifacts stay under the versioned output directory; archive also preserves failures and logs.


In [ ]:
run('reference_v3.review','--output',str(OUT))
run('reference_v3.plots','--output',str(OUT))
ZIP=package()
print('Artifacts:',OUT)
print('ZIP:',ZIP)
print(json.loads((OUT/'CERTIFICATION_RESULTS.json').read_text())['metadata'])
# Optional manual download, especially when USE_DRIVE=False:
# from google.colab import files
# files.download(ZIP)
